# Phase 3 v2 — Forecasting, Scenario Stress Tests, and Expanded Charts

This notebook upgrades Phase 3 in four ways:

1. **Uses the stronger Phase 2 logic** by preferring the better-performing **Extra Trees** setup with the **annualized 3-year target** (`target_pop_change_3y_avg`) instead of the noisier 1-year recursive target.
2. **Keeps municipal / regional / island analysis** with municipality and region as categorical geography inputs.
3. **Strengthens hazard scenario encoding**, especially for earthquakes, so the earthquake stress test is more distinguishable from baseline when appropriate.
4. **Generates more charts** after the scenario runs, including:
   - side-by-side island fan charts,
   - scenario median path comparison,
   - island impact-over-time chart,
   - region 2030 impact bar chart,
   - top/bottom municipality impact bar charts,
   - region-year heatmap of median impacts,
   - uncertainty-width comparison chart.

## Interpretation note
These are **model-based scenario stress tests**, not literal physical forecasts. The hazard scenario inputs are stylized mappings into the engineered disaster features, so the results are most useful for **relative comparison**:
- baseline vs hurricane,
- baseline vs earthquake,
- high vs low vulnerability contexts,
- municipal / regional / island differences.


In [ ]:

# Core imports
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA

np.random.seed(42)
pd.set_option("display.max_columns", 200)


In [ ]:

# Output folder and helpers
OUTDIR = Path("phase3_outputs_v2")
OUTDIR.mkdir(exist_ok=True, parents=True)

saved_files = []

def save_csv(frame, name):
    path = OUTDIR / name
    frame.to_csv(path, index=False)
    saved_files.append({"file": name, "path": str(path.resolve()), "type": "csv"})
    print(f"Saved CSV: {path.resolve()}")
    return path

def save_json(obj, name):
    path = OUTDIR / name
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    saved_files.append({"file": name, "path": str(path.resolve()), "type": "json"})
    print(f"Saved JSON: {path.resolve()}")
    return path

def save_figure(fig, name, dpi=160):
    path = OUTDIR / name
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    saved_files.append({"file": name, "path": str(path.resolve()), "type": "figure"})
    print(f"Saved FIG: {path.resolve()}")
    return path


In [ ]:

# Load Phase 1 rebuild data
candidate_files = [
    Path("processed_puerto_rico_data_rebuild.csv"),
    Path("processed_puerto_rico_data_enriched.csv"),
    Path("/mnt/data/processed_puerto_rico_data_rebuild.csv"),
    Path("/mnt/data/processed_puerto_rico_data_enriched.csv"),
]

data_path = None
for p in candidate_files:
    if p.exists():
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Could not find processed Puerto Rico Phase 1 dataset.")

df = pd.read_csv(data_path)
print("Loaded:", data_path)
print("Shape:", df.shape)
print("Years:", int(df["year"].min()), "to", int(df["year"].max()))
print("Municipalities:", int(df["municipio"].nunique()))


In [ ]:

# Region mapping
region_map = {
    "San Juan": "Metro","Bayamón": "Metro","Carolina": "Metro","Cataño": "Metro","Guaynabo": "Metro","Toa Alta": "Metro","Toa Baja": "Metro","Trujillo Alto": "Metro",
    "Arecibo": "North","Barceloneta": "North","Camuy": "North","Dorado": "North","Florida": "North","Hatillo": "North","Manatí": "North","Quebradillas": "North","Vega Alta": "North","Vega Baja": "North",
    "Arroyo": "South","Coamo": "South","Guayama": "South","Guayanilla": "South","Juana Díaz": "South","Patillas": "South","Peñuelas": "South","Ponce": "South","Salinas": "South","Santa Isabel": "South","Villalba": "South","Yauco": "South",
    "Aguada": "West","Aguadilla": "West","Añasco": "West","Cabo Rojo": "West","Guánica": "West","Hormigueros": "West","Isabela": "West","Lajas": "West","Las Marías": "West","Maricao": "West","Mayagüez": "West","Moca": "West","Rincón": "West","Sabana Grande": "West","San Germán": "West","San Sebastián": "West",
    "Canóvanas": "East","Ceiba": "East","Fajardo": "East","Humacao": "East","Juncos": "East","Las Piedras": "East","Loíza": "East","Luquillo": "East","Maunabo": "East","Naguabo": "East","Río Grande": "East","San Lorenzo": "East","Yabucoa": "East","Caguas": "East","Gurabo": "East","Culebra": "East","Vieques": "East",
    "Adjuntas": "Central Mountains","Aguas Buenas": "Central Mountains","Aibonito": "Central Mountains","Barranquitas": "Central Mountains","Cayey": "Central Mountains","Ciales": "Central Mountains","Cidra": "Central Mountains","Comerío": "Central Mountains","Corozal": "Central Mountains","Jayuya": "Central Mountains","Lares": "Central Mountains","Morovis": "Central Mountains","Naranjito": "Central Mountains","Orocovis": "Central Mountains","Utuado": "Central Mountains",
}

df["region"] = df["municipio"].map(region_map).fillna("Other")
print(df["region"].value_counts(dropna=False))


## Rebuild / verify PCA-based SVI if needed

In [ ]:

svi_indicator_columns = [
    "poverty_rate_pct","unemployment_rate_pct","per_capita_income_inv","no_hs_diploma_pct","under_18_pct",
    "over_65_pct","disability_pct","single_parent_pct","minority_pct","limited_english_pct",
    "multi_unit_housing_pct","mobile_homes_pct","crowding_pct","no_vehicle_pct","pct_group_quarters",
]

missing_svi_cols = [c for c in svi_indicator_columns if c not in df.columns]
if missing_svi_cols:
    raise ValueError(f"Missing required SVI indicator columns: {missing_svi_cols}")

if "svi_pca_1" not in df.columns:
    svi_tmp = df[svi_indicator_columns].apply(pd.to_numeric, errors="coerce")
    svi_imp = pd.DataFrame(
        SimpleImputer(strategy="median").fit_transform(svi_tmp),
        columns=svi_indicator_columns,
        index=df.index
    )
    pca = PCA(n_components=1, random_state=42)
    df["svi_pca_1"] = pca.fit_transform(svi_imp).ravel()

if "wind_3yr_x_svi" not in df.columns:
    df["wind_3yr_x_svi"] = df["wind_3yr_sum"] * df["svi_pca_1"]
if "seismic_3yr_x_svi" not in df.columns:
    df["seismic_3yr_x_svi"] = df["seismic_3yr_sum"] * df["svi_pca_1"]


## Model setup: stronger Phase 2 backbone

This Phase 3 v2 notebook uses the **annualized 3-year target** as the primary forecasting signal:

- `target_pop_change_3y_avg`

The forecast loop then interprets this annualized value as the model-implied annual growth rate for recursive scenario simulation. This is a pragmatic compromise that usually behaves more smoothly than the noisier 1-year target.


In [ ]:

TARGET = "target_pop_change_3y_avg"

numeric_features = [
    "lag_pop_change_1",
    "lag_pop_change_2",
    "svi_pca_1",
    "wind_3yr_sum",
    "seismic_3yr_sum",
    "wind_3yr_x_svi",
    "seismic_3yr_x_svi",
    "income_growth",
    "employment_proxy_growth" if "employment_proxy_growth" in df.columns else None,
    "establishment_growth",
    "lag_total_crime_rate",
    "year",
]

numeric_features = [c for c in numeric_features if c is not None]
categorical_features = ["municipio", "region"]
all_features = numeric_features + categorical_features

required_cols = list(set(all_features + [TARGET, "total_population"]))
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns for Phase 3 v2: {missing}")

model_df = df[all_features + [TARGET, "total_population"]].copy()
model_df = model_df.dropna(subset=[TARGET, "total_population"]).reset_index(drop=True)

print("Modeling rows:", len(model_df))
print("Target:", TARGET)
print("Features:", all_features)


In [ ]:

# Train / validation / test diagnostic split
train_mask = model_df["year"] <= 2020
val_mask = model_df["year"].between(2021, 2022)
test_mask = model_df["year"] >= 2023

X_train = model_df.loc[train_mask, all_features]
y_train = model_df.loc[train_mask, TARGET]

X_val = model_df.loc[val_mask, all_features]
y_val = model_df.loc[val_mask, TARGET]

X_test = model_df.loc[test_mask, all_features]
y_test = model_df.loc[test_mask, TARGET]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

phase3_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(
        n_estimators=500,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

phase3_model.fit(X_train, y_train)

def metric_block(y_true, y_pred):
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }

val_pred = phase3_model.predict(X_val)
test_pred = phase3_model.predict(X_test)

phase3_v2_metrics = {
    "validation": metric_block(y_val, val_pred),
    "test": metric_block(y_test, test_pred),
}

phase3_v2_metrics


In [ ]:

# Fit final model on all available rows for scenario forecasting
phase3_model.fit(model_df[all_features], model_df[TARGET])
all_fitted = phase3_model.predict(model_df[all_features])
residuals = model_df[TARGET].values - all_fitted

phase3_v2_summary = {
    "phase": "Phase 3 v2",
    "model": "ExtraTreesRegressor",
    "target": TARGET,
    "features": all_features,
    "diagnostics": phase3_v2_metrics,
    "residual_mean": float(np.mean(residuals)),
    "residual_std": float(np.std(residuals)),
    "n_rows_final_fit": int(len(model_df)),
}
save_json(phase3_v2_summary, "phase3_v2_model_summary.json")
phase3_v2_summary


## Scenario controls

You can safely start with:
- `n_sims = 100`
- `save_detail = False`

Then raise simulation counts later for final figures.


In [ ]:

shock_year = 2026
hurricane_category = 3
earthquake_magnitude = 6.8
forecast_start_year = int(df["year"].max()) + 1
forecast_end_year = 2030

n_sims = 100
save_detail = False
random_seed = 42

# Stronger stylized mappings than v1.
# These mappings are intentionally a bit more forceful so the hazard scenarios
# are easier to distinguish from baseline in the model-based stress tests.
category_to_wind_score = {1: 85.0, 2: 105.0, 3: 130.0, 4: 160.0, 5: 195.0}

def magnitude_to_seismic_score(mag):
    # More forceful than v1, while still monotonic and simple.
    return max(0.0, (mag - 4.5)) ** 2 * 22.0

scenario_definitions = {
    "baseline_no_disaster": {
        "type": "none",
        "shock_year": None,
        "wind_score": 0.0,
        "quake_score": 0.0,
        "label": "No natural disaster",
    },
    f"hurricane_cat{hurricane_category}_{shock_year}": {
        "type": "hurricane",
        "shock_year": shock_year,
        "wind_score": category_to_wind_score[hurricane_category],
        "quake_score": 0.0,
        "label": f"Hurricane Category {hurricane_category} ({shock_year})",
    },
    f"earthquake_m{str(earthquake_magnitude).replace('.', '')}_{shock_year}": {
        "type": "earthquake",
        "shock_year": shock_year,
        "wind_score": 0.0,
        "quake_score": magnitude_to_seismic_score(earthquake_magnitude),
        "label": f"Earthquake M{earthquake_magnitude} ({shock_year})",
    },
}
scenario_definitions


In [ ]:

# Forecast seed rows and reference tables
latest_year = int(df["year"].max())
latest_rows = (
    df.loc[df["year"] == latest_year]
      .sort_values("municipio")
      .reset_index(drop=True)
      .copy()
)

exog_cols = [c for c in ["income_growth", "employment_proxy_growth", "establishment_growth", "lag_total_crime_rate", "svi_pca_1"] if c in df.columns]
exog_by_muni = df.groupby("municipio")[exog_cols].mean().reset_index()
latest_rows = latest_rows.drop(columns=[c for c in exog_cols if c in latest_rows.columns], errors="ignore")
latest_rows = latest_rows.merge(exog_by_muni, on="municipio", how="left")

historical_island = (
    df.groupby("year", as_index=False)["total_population"]
      .sum()
      .rename(columns={"total_population": "population"})
)
historical_region = (
    df.groupby(["region", "year"], as_index=False)["total_population"]
      .sum()
      .rename(columns={"total_population": "population"})
)


In [ ]:

# Forecast helper logic
def initialize_histories(frame):
    histories = {}
    for muni, g in frame.groupby("municipio", sort=False):
        g = g.sort_values("year").copy()
        histories[muni] = {
            "pop": list(g["total_population"].tail(4).astype(float).values),
            "wind": list(g["wind_3yr_sum"].tail(3).fillna(0).astype(float).values),
            "quake": list(g["seismic_3yr_sum"].tail(3).fillna(0).astype(float).values),
            "svi": float(g["svi_pca_1"].iloc[-1]),
            "region": g["region"].iloc[-1],
            "base": g.iloc[-1].to_dict(),
        }
    return histories

def safe_mean(vals, fallback=0.0):
    vals = [float(v) for v in vals if pd.notna(v)]
    if len(vals) == 0:
        return fallback
    return float(np.mean(vals))

def build_feature_row(muni, hist, year, scenario):
    pop_hist = hist["pop"]
    wind_hist = hist["wind"]
    quake_hist = hist["quake"]

    lag1 = np.nan
    lag2 = np.nan
    if len(pop_hist) >= 2 and pop_hist[-2] not in [0, np.nan]:
        lag1 = ((pop_hist[-1] / pop_hist[-2]) - 1.0) * 100.0
    if len(pop_hist) >= 3 and pop_hist[-3] not in [0, np.nan]:
        lag2 = ((pop_hist[-2] / pop_hist[-3]) - 1.0) * 100.0

    current_wind = safe_mean(wind_hist[-3:], fallback=0.0)
    current_quake = safe_mean(quake_hist[-3:], fallback=0.0)

    if scenario["shock_year"] is not None and year == scenario["shock_year"]:
        current_wind += scenario["wind_score"]
        current_quake += scenario["quake_score"]

    svi = hist["svi"]
    base = hist["base"]

    row = {
        "lag_pop_change_1": lag1,
        "lag_pop_change_2": lag2,
        "svi_pca_1": svi,
        "wind_3yr_sum": current_wind,
        "seismic_3yr_sum": current_quake,
        "wind_3yr_x_svi": current_wind * svi,
        "seismic_3yr_x_svi": current_quake * svi,
        "income_growth": base.get("income_growth", np.nan),
        "employment_proxy_growth": base.get("employment_proxy_growth", np.nan),
        "establishment_growth": base.get("establishment_growth", np.nan),
        "lag_total_crime_rate": base.get("lag_total_crime_rate", np.nan),
        "year": year,
        "municipio": muni,
        "region": hist["region"],
    }
    return row

def recursive_scenario_forecast(model, latest_rows, scenario_name, scenario, residuals, n_sims=100, save_detail=False, random_seed=42):
    rng = np.random.default_rng(random_seed)
    histories = initialize_histories(df)
    muni_order = list(histories.keys())
    forecast_years = list(range(forecast_start_year, forecast_end_year + 1))

    detail_records = []
    island_records = []
    region_records = []
    muni_records = []

    # residual draws create fan-chart dispersion
    residual_scale = max(np.std(residuals), 0.25)

    for sim in range(n_sims):
        sim_histories = {
            k: {
                "pop": list(v["pop"]),
                "wind": list(v["wind"]),
                "quake": list(v["quake"]),
                "svi": v["svi"],
                "region": v["region"],
                "base": dict(v["base"]),
            }
            for k, v in histories.items()
        }

        for year in forecast_years:
            rows = []
            for muni in muni_order:
                rows.append(build_feature_row(muni, sim_histories[muni], year, scenario))
            X_future = pd.DataFrame(rows)

            # Keep only modeling columns and create missing columns if needed
            for col in all_features:
                if col not in X_future.columns:
                    X_future[col] = np.nan
            X_future = X_future[all_features]

            pred_rate = model.predict(X_future)
            pred_rate = pred_rate + rng.normal(0.0, residual_scale, size=len(pred_rate))

            next_pops = []
            for i, muni in enumerate(muni_order):
                hist = sim_histories[muni]
                current_pop = float(hist["pop"][-1])
                annual_rate = float(pred_rate[i]) / 100.0
                next_pop = max(1.0, current_pop * (1.0 + annual_rate))
                next_pops.append(next_pop)

                # update histories
                hist["pop"].append(next_pop)
                hist["pop"] = hist["pop"][-4:]

                new_wind = rows[i]["wind_3yr_sum"]
                new_quake = rows[i]["seismic_3yr_sum"]
                hist["wind"].append(new_wind)
                hist["wind"] = hist["wind"][-3:]
                hist["quake"].append(new_quake)
                hist["quake"] = hist["quake"][-3:]

                if save_detail:
                    detail_records.append({
                        "scenario_name": scenario_name,
                        "scenario_label": scenario["label"],
                        "simulation": sim,
                        "year": year,
                        "municipio": muni,
                        "region": hist["region"],
                        "predicted_population": next_pop,
                        "predicted_growth_rate_pct": pred_rate[i],
                    })

            temp = pd.DataFrame({
                "municipio": muni_order,
                "region": [sim_histories[m]["region"] for m in muni_order],
                "population": next_pops,
            })
            island_pop = float(temp["population"].sum())
            island_records.append({
                "scenario_name": scenario_name,
                "scenario_label": scenario["label"],
                "simulation": sim,
                "year": year,
                "island_population": island_pop,
            })

            region_agg = temp.groupby("region", as_index=False)["population"].sum()
            for _, r in region_agg.iterrows():
                region_records.append({
                    "scenario_name": scenario_name,
                    "scenario_label": scenario["label"],
                    "simulation": sim,
                    "year": year,
                    "region": r["region"],
                    "region_population": r["population"],
                })

            for _, r in temp.iterrows():
                muni_records.append({
                    "scenario_name": scenario_name,
                    "scenario_label": scenario["label"],
                    "simulation": sim,
                    "year": year,
                    "municipio": r["municipio"],
                    "region": r["region"],
                    "municipio_population": r["population"],
                })

    detail_df = pd.DataFrame(detail_records) if save_detail else pd.DataFrame()
    island_sim = pd.DataFrame(island_records)
    region_sim = pd.DataFrame(region_records)
    muni_sim = pd.DataFrame(muni_records)
    return detail_df, island_sim, region_sim, muni_sim


In [ ]:

# Run scenarios
scenario_outputs = {}
scenario_timing = []
overall_t0 = time.time()

for idx, (scen_name, scen) in enumerate(scenario_definitions.items(), start=1):
    t0 = time.time()
    print(f"Running scenario {idx}/{len(scenario_definitions)}: {scen_name}")

    detail_df, island_sim, region_sim, muni_sim = recursive_scenario_forecast(
        model=phase3_model,
        latest_rows=latest_rows,
        scenario_name=scen_name,
        scenario=scen,
        residuals=residuals,
        n_sims=n_sims,
        save_detail=save_detail,
        random_seed=random_seed + idx,
    )

    scenario_outputs[scen_name] = {
        "detail": detail_df,
        "island_sim": island_sim,
        "region_sim": region_sim,
        "muni_sim": muni_sim,
    }
    scenario_timing.append({
        "scenario_name": scen_name,
        "seconds": time.time() - t0,
        "n_sims": n_sims,
    })
    print(f"Finished {scen_name} in {time.time() - t0:.2f} seconds")

scenario_timing_df = pd.DataFrame(scenario_timing)
save_csv(scenario_timing_df, "phase3_v2_scenario_timing.csv")
print(f"Total runtime: {time.time() - overall_t0:.2f} seconds")


In [ ]:

# Summaries
def summarize_distribution(sim_df, value_col, group_cols):
    out = (
        sim_df.groupby(group_cols)[value_col]
        .quantile([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
        .unstack()
        .reset_index()
    )
    out.columns = group_cols + ["p05", "p10", "p25", "p50", "p75", "p90", "p95"]
    return out

island_summaries = []
region_summaries = []
municipal_summaries = []

for scen_name, bundle in scenario_outputs.items():
    island_summaries.append(
        summarize_distribution(
            bundle["island_sim"],
            "island_population",
            ["scenario_name", "scenario_label", "year"]
        )
    )
    region_summaries.append(
        summarize_distribution(
            bundle["region_sim"],
            "region_population",
            ["scenario_name", "scenario_label", "region", "year"]
        )
    )
    municipal_summaries.append(
        summarize_distribution(
            bundle["muni_sim"],
            "municipio_population",
            ["scenario_name", "scenario_label", "municipio", "region", "year"]
        )
    )

phase3_v2_island_scenario_summary = pd.concat(island_summaries, ignore_index=True)
phase3_v2_region_scenario_summary = pd.concat(region_summaries, ignore_index=True)
phase3_v2_municipal_scenario_summary = pd.concat(municipal_summaries, ignore_index=True)

save_csv(phase3_v2_island_scenario_summary, "phase3_v2_island_scenario_summary.csv")
save_csv(phase3_v2_region_scenario_summary, "phase3_v2_region_scenario_summary.csv")
save_csv(phase3_v2_municipal_scenario_summary, "phase3_v2_municipal_scenario_summary.csv")


In [ ]:

# Impact summaries against baseline
baseline_name = "baseline_no_disaster"

baseline_island = phase3_v2_island_scenario_summary[
    phase3_v2_island_scenario_summary["scenario_name"] == baseline_name
][["year", "p05", "p10", "p25", "p50", "p75", "p90", "p95"]].rename(
    columns={c: f"baseline_{c}" for c in ["p05", "p10", "p25", "p50", "p75", "p90", "p95"]}
)

island_impact = phase3_v2_island_scenario_summary.merge(baseline_island, on="year", how="left")
for q in ["p05", "p10", "p25", "p50", "p75", "p90", "p95"]:
    island_impact[f"{q}_vs_baseline"] = island_impact[q] - island_impact[f"baseline_{q}"]

save_csv(island_impact, "phase3_v2_island_impact_summary.csv")

region_baseline = phase3_v2_region_scenario_summary[
    phase3_v2_region_scenario_summary["scenario_name"] == baseline_name
][["region", "year", "p50"]].rename(columns={"p50": "baseline_p50"})

region_impact = phase3_v2_region_scenario_summary.merge(region_baseline, on=["region", "year"], how="left")
region_impact["median_vs_baseline"] = region_impact["p50"] - region_impact["baseline_p50"]
save_csv(region_impact, "phase3_v2_region_impact_summary.csv")

municipal_baseline = phase3_v2_municipal_scenario_summary[
    phase3_v2_municipal_scenario_summary["scenario_name"] == baseline_name
][["municipio", "year", "p50"]].rename(columns={"p50": "baseline_p50"})

municipal_impact = phase3_v2_municipal_scenario_summary.merge(
    municipal_baseline, on=["municipio", "year"], how="left"
)
municipal_impact["median_vs_baseline"] = municipal_impact["p50"] - municipal_impact["baseline_p50"]
save_csv(municipal_impact, "phase3_v2_municipal_impact_summary.csv")


## Plotting helpers and derived tables

In [ ]:

def get_island_summary(name):
    return phase3_v2_island_scenario_summary[
        phase3_v2_island_scenario_summary["scenario_name"] == name
    ].sort_values("year").copy()

def get_region_2030_for_scenario(name):
    tmp = region_impact[(region_impact["scenario_name"] == name) & (region_impact["year"] == forecast_end_year)].copy()
    return tmp.sort_values("median_vs_baseline")

def get_muni_2030_for_scenario(name):
    tmp = municipal_impact[(municipal_impact["scenario_name"] == name) & (municipal_impact["year"] == forecast_end_year)].copy()
    return tmp.sort_values("median_vs_baseline")

def plot_fan(ax, hist_df, scen_df, title):
    ax.plot(hist_df["year"], hist_df["population"], marker="o", linewidth=1.6, label="Observed")
    ax.fill_between(scen_df["year"], scen_df["p05"], scen_df["p95"], alpha=0.12, label="5%-95%")
    ax.fill_between(scen_df["year"], scen_df["p10"], scen_df["p90"], alpha=0.18, label="10%-90%")
    ax.fill_between(scen_df["year"], scen_df["p25"], scen_df["p75"], alpha=0.24, label="25%-75%")
    ax.plot(scen_df["year"], scen_df["p50"], linewidth=2.2, label="Median forecast")
    ax.axvline(latest_year, linestyle="--", linewidth=1.2)
    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("Population")
    ax.grid(alpha=0.25)

def plot_compare_panel(ax, hist_df, baseline_df, scenario_df, title):
    ax.plot(hist_df["year"], hist_df["population"], marker="o", linewidth=1.5, label="Observed")
    ax.fill_between(baseline_df["year"], baseline_df["p25"], baseline_df["p75"], alpha=0.18, label="Baseline 25%-75%")
    ax.plot(baseline_df["year"], baseline_df["p50"], linestyle="--", linewidth=2.0, label="Baseline median")
    ax.fill_between(scenario_df["year"], scenario_df["p25"], scenario_df["p75"], alpha=0.22, label="Scenario 25%-75%")
    ax.plot(scenario_df["year"], scenario_df["p50"], linewidth=2.2, label="Scenario median")
    ax.axvline(latest_year, linestyle="--", linewidth=1.2)
    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("Population")
    ax.grid(alpha=0.25)


In [ ]:

baseline_name = "baseline_no_disaster"
hurricane_name = f"hurricane_cat{hurricane_category}_{shock_year}"
earthquake_name = f"earthquake_m{str(earthquake_magnitude).replace('.', '')}_{shock_year}"

baseline_df = get_island_summary(baseline_name)
hurricane_df = get_island_summary(hurricane_name)
earthquake_df = get_island_summary(earthquake_name)


In [ ]:

# Chart 1: side-by-side baseline vs hazard fan charts
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_compare_panel(
    axes[0], historical_island, baseline_df, hurricane_df,
    f"No disaster vs Hurricane Category {hurricane_category} ({shock_year})"
)
plot_compare_panel(
    axes[1], historical_island, baseline_df, earthquake_df,
    f"No disaster vs Earthquake M{earthquake_magnitude} ({shock_year})"
)
fig.suptitle("Scenario comparison fan charts for Puerto Rico island-level population", fontsize=16)
for ax in axes: ax.legend(loc="best")
fig.tight_layout()
save_figure(fig, "phase3_v2_side_by_side_baseline_vs_hazard.png")
plt.show()


In [ ]:

# Chart 2: scenario median paths only
fig = plt.figure(figsize=(10, 5))
ax = plt.gca()
ax.plot(historical_island["year"], historical_island["population"], color="black", marker="o", linewidth=1.5, label="Observed")
ax.plot(baseline_df["year"], baseline_df["p50"], linestyle="--", linewidth=2.2, label="Baseline median")
ax.plot(hurricane_df["year"], hurricane_df["p50"], linewidth=2.2, label=f"Hurricane Cat {hurricane_category} median")
ax.plot(earthquake_df["year"], earthquake_df["p50"], linewidth=2.2, label=f"Earthquake M{earthquake_magnitude} median")
ax.axvline(latest_year, linestyle="--", linewidth=1.2)
ax.set_title("Island-level scenario median forecast paths")
ax.set_xlabel("Year")
ax.set_ylabel("Population")
ax.grid(alpha=0.25)
ax.legend(loc="best")
fig.tight_layout()
save_figure(fig, "phase3_v2_island_median_paths.png")
plt.show()


In [ ]:

# Chart 3: island impact over time relative to baseline median
impact_plot = island_impact[island_impact["scenario_name"] != baseline_name].copy()

fig = plt.figure(figsize=(10, 5))
ax = plt.gca()
for scen_name, g in impact_plot.groupby("scenario_label"):
    ax.plot(g["year"], g["p50_vs_baseline"], marker="o", linewidth=2, label=scen_name)
ax.axhline(0, linestyle="--", linewidth=1.2)
ax.set_title("Island-level median impact relative to baseline")
ax.set_xlabel("Year")
ax.set_ylabel("Median population difference vs baseline")
ax.grid(alpha=0.25)
ax.legend(loc="best")
fig.tight_layout()
save_figure(fig, "phase3_v2_island_median_impact_over_time.png")
plt.show()


In [ ]:

# Chart 4: region 2030 impacts by scenario
region_2030_h = get_region_2030_for_scenario(hurricane_name)
region_2030_q = get_region_2030_for_scenario(earthquake_name)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharex=False)
axes[0].barh(region_2030_h["region"], region_2030_h["median_vs_baseline"])
axes[0].axvline(0, linestyle="--", linewidth=1.2)
axes[0].set_title(f"Region 2030 impact vs baseline\nHurricane Category {hurricane_category}")
axes[0].set_xlabel("Median population difference")

axes[1].barh(region_2030_q["region"], region_2030_q["median_vs_baseline"])
axes[1].axvline(0, linestyle="--", linewidth=1.2)
axes[1].set_title(f"Region 2030 impact vs baseline\nEarthquake M{earthquake_magnitude}")
axes[1].set_xlabel("Median population difference")

fig.tight_layout()
save_figure(fig, "phase3_v2_region_2030_impact_bars.png")
plt.show()


In [ ]:

# Chart 5: top/bottom municipal impacts in 2030
def top_bottom(frame, n=12):
    return pd.concat([frame.head(n), frame.tail(n)], ignore_index=True)

muni_2030_h = top_bottom(get_muni_2030_for_scenario(hurricane_name), 10)
muni_2030_q = top_bottom(get_muni_2030_for_scenario(earthquake_name), 10)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].barh(muni_2030_h["municipio"], muni_2030_h["median_vs_baseline"])
axes[0].axvline(0, linestyle="--", linewidth=1.2)
axes[0].set_title(f"Most / least affected municipios in 2030\nHurricane Category {hurricane_category}")
axes[0].set_xlabel("Median population difference vs baseline")

axes[1].barh(muni_2030_q["municipio"], muni_2030_q["median_vs_baseline"])
axes[1].axvline(0, linestyle="--", linewidth=1.2)
axes[1].set_title(f"Most / least affected municipios in 2030\nEarthquake M{earthquake_magnitude}")
axes[1].set_xlabel("Median population difference vs baseline")

fig.tight_layout()
save_figure(fig, "phase3_v2_municipal_2030_top_bottom_impact_bars.png")
plt.show()


In [ ]:

# Chart 6: region-year heatmap of hurricane impact vs baseline
h_region = region_impact[region_impact["scenario_name"] == hurricane_name].copy()
heat_h = h_region.pivot(index="region", columns="year", values="median_vs_baseline").sort_index()

fig = plt.figure(figsize=(10, 5))
ax = plt.gca()
im = ax.imshow(heat_h.values, aspect="auto")
ax.set_xticks(range(len(heat_h.columns)))
ax.set_xticklabels(list(heat_h.columns))
ax.set_yticks(range(len(heat_h.index)))
ax.set_yticklabels(list(heat_h.index))
ax.set_title(f"Region-year median impact heatmap vs baseline\nHurricane Category {hurricane_category}")
ax.set_xlabel("Year")
ax.set_ylabel("Region")
fig.colorbar(im, ax=ax, label="Median population difference vs baseline")
fig.tight_layout()
save_figure(fig, "phase3_v2_region_year_heatmap_hurricane.png")
plt.show()


In [ ]:

# Chart 7: uncertainty width comparison (p95 - p05)
fig = plt.figure(figsize=(10, 5))
ax = plt.gca()
for name, label, frame in [
    (baseline_name, "Baseline", baseline_df),
    (hurricane_name, f"Hurricane Cat {hurricane_category}", hurricane_df),
    (earthquake_name, f"Earthquake M{earthquake_magnitude}", earthquake_df),
]:
    tmp = frame.copy()
    tmp["width_90"] = tmp["p95"] - tmp["p05"]
    ax.plot(tmp["year"], tmp["width_90"], marker="o", linewidth=2, label=label)

ax.set_title("Island-level forecast uncertainty width (p95 - p05)")
ax.set_xlabel("Year")
ax.set_ylabel("Population interval width")
ax.grid(alpha=0.25)
ax.legend(loc="best")
fig.tight_layout()
save_figure(fig, "phase3_v2_uncertainty_width_comparison.png")
plt.show()


In [ ]:

# Chart 8: direct fan charts for each scenario
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_fan(axes[0], historical_island, baseline_df, "Baseline (no natural disaster)")
plot_fan(axes[1], historical_island, hurricane_df, f"Hurricane Category {hurricane_category}")
plot_fan(axes[2], historical_island, earthquake_df, f"Earthquake M{earthquake_magnitude}")
for ax in axes: ax.legend(loc="best")
fig.suptitle("Island-level fan charts by scenario", fontsize=16)
fig.tight_layout()
save_figure(fig, "phase3_v2_island_fan_charts_all_scenarios.png")
plt.show()


In [ ]:

# Save compact interpretation tables
island_interpretation = island_impact[
    island_impact["year"].isin([forecast_start_year, shock_year, forecast_end_year])
].copy()
save_csv(island_interpretation, "phase3_v2_island_interpretation_table.csv")

region_2030_compare = region_impact[region_impact["year"] == forecast_end_year].copy()
save_csv(region_2030_compare, "phase3_v2_region_2030_comparison.csv")

municipal_2030_compare = municipal_impact[municipal_impact["year"] == forecast_end_year].copy()
save_csv(municipal_2030_compare, "phase3_v2_municipal_2030_comparison.csv")

phase3_chart_index = pd.DataFrame({
    "chart_file": [
        "phase3_v2_side_by_side_baseline_vs_hazard.png",
        "phase3_v2_island_median_paths.png",
        "phase3_v2_island_median_impact_over_time.png",
        "phase3_v2_region_2030_impact_bars.png",
        "phase3_v2_municipal_2030_top_bottom_impact_bars.png",
        "phase3_v2_region_year_heatmap_hurricane.png",
        "phase3_v2_uncertainty_width_comparison.png",
        "phase3_v2_island_fan_charts_all_scenarios.png",
    ],
    "description": [
        "Baseline vs hurricane and baseline vs earthquake side-by-side fan charts",
        "Observed plus scenario median forecast paths",
        "Island-level median impact relative to baseline across years",
        "Regional 2030 population impacts relative to baseline",
        "Top/bottom municipios by 2030 scenario impact relative to baseline",
        "Heatmap of regional hurricane impacts by year",
        "Comparison of uncertainty band widths across scenarios",
        "Standalone island fan charts for baseline, hurricane, and earthquake",
    ]
})
save_csv(phase3_chart_index, "phase3_v2_chart_index.csv")


In [ ]:

# Save manifest
manifest_df = pd.DataFrame(saved_files)
save_csv(manifest_df, "phase3_v2_saved_files_manifest.csv")
manifest_df
